[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/34_representation_learning/03_contrastive_and_self_supervised.ipynb)

# Bài 03 — Học tương phản và tự giám sát

**🎯 Mục tiêu**
- **Self-supervised learning**: tự tạo tín hiệu học từ dữ liệu không nhãn — và vì sao mục tiêu tái tạo (bài 01) chưa đủ.
- **Học tương phản** (SimCLR): hai góc nhìn tăng cường của cùng ảnh là cặp dương; **loss InfoNCE / NT-Xent** cài from scratch.
- Đánh giá biểu diễn bằng **linear probe** với rất ít nhãn: so với pixel, encoder ngẫu nhiên, autoencoder, và train có giám sát từ đầu.
- **Augmentation quyết định biểu diễn** bất biến với cái gì (ablation).
- Bức tranh rộng: BYOL/SimSiam, DINO, MAE, CLIP — và embedding trong thực tế.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/34_representation_learning

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from ml_course.datasets import SHAPE_CLASSES, make_shapes
from ml_course.visualization import set_style

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
to_tensor = lambda X: torch.tensor(X[:, None], dtype=torch.float32)
X_unlabeled, y_hidden = make_shapes(20000, seed=0)                     # y_hidden chỉ dùng để đánh giá
X_test, y_test = make_shapes(2000, seed=1)
X_unlabeled_t, X_test_t = to_tensor(X_unlabeled), to_tensor(X_test)
print("thiết bị:", device)

## 1. Ý tưởng: dạy mô hình điều gì là "giống nhau"

Bài 01: autoencoder học tái tạo pixel → latent ưu tiên **vị trí và kích thước**, gần như bỏ qua **loại hình** — vì mục tiêu không nói gì về điều ta quan tâm. Học tương phản đổi câu hỏi: thay vì "tái tạo ảnh", mô hình phải nhận ra **hai phiên bản biến đổi của cùng một ảnh** giữa nhiều ảnh khác.

**SimCLR** (Chen et al., 2020):
1. Mỗi ảnh $x$ trong batch → hai **góc nhìn** $\tilde x_1, \tilde x_2$ bằng augmentation ngẫu nhiên (xoay, dịch, lật, độ sáng, nhiễu).
2. Encoder $f$ → biểu diễn $h$; **projection head** $g$ (MLP nhỏ) → $z$ (chuẩn hoá về độ dài 1).
3. Loss **NT-Xent / InfoNCE**: với mỗi góc nhìn, coi góc nhìn còn lại của cùng ảnh là lớp đúng trong một bài phân loại giữa $2N - 1$ ứng viên:

$$\ell_i = -\log\frac{\exp(\text{sim}(z_i, z_{j(i)})/\tau)}{\sum_{k\ne i}\exp(\text{sim}(z_i, z_k)/\tau)}$$

($\tau$: nhiệt độ). Muốn hai góc nhìn khớp nhau mà không khớp với ảnh khác, $h$ phải giữ những gì **không đổi qua augmentation** (loại hình) và bỏ những gì augmentation làm thay đổi (vị trí, góc xoay, độ sáng).

In [ ]:
def random_affine(x, g, max_rot=math.pi, max_shift=0.15):
    B = x.shape[0]
    angle = (torch.rand(B, generator=g) * 2 - 1) * max_rot
    shift = (torch.rand(2, B, generator=g) * 2 - 1) * max_shift * 2
    flip = torch.where(torch.rand(B, generator=g) < 0.5, -1.0, 1.0)
    cos, sin = angle.cos(), angle.sin()
    theta = torch.stack([torch.stack([cos * flip, -sin, shift[0]], 1), torch.stack([sin * flip, cos, shift[1]], 1)], 1)
    return F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), align_corners=False)


def augment(x, g, geometric=True):
    if geometric:
        x = random_affine(x, g)
    brightness = 0.6 + 0.8 * torch.rand(x.shape[0], 1, 1, 1, generator=g)
    return (x * brightness + 0.15 * torch.randn(x.shape, generator=g)).clamp(0, 1)


g_demo = torch.Generator().manual_seed(0)
fig, axes = plt.subplots(2, 6, figsize=(11, 3.8))
for j in range(6):
    v1, v2 = augment(X_unlabeled_t[j:j + 1], g_demo), augment(X_unlabeled_t[j:j + 1], g_demo)
    axes[0, j].imshow(v1[0, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, j].imshow(v2[0, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, j].set_title(f"ảnh {j}: góc nhìn 1", fontsize=8)
    axes[1, j].set_title("góc nhìn 2", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def nt_xent(z1, z2, tau=0.2):
    z = F.normalize(torch.cat([z1, z2]), dim=1)                        # (2N, d), độ dài 1 → tích vô hướng = cosine
    sim = z @ z.T / tau
    sim.fill_diagonal_(float("-inf"))                                  # không so một mẫu với chính nó
    n = len(z1)
    positives = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)   # i ↔ i + N
    return F.cross_entropy(sim, positives)


z = torch.randn(4, 8)
print(f"NT-Xent khi hai góc nhìn giống hệt: {nt_xent(z, z):.3f} | khi ngẫu nhiên: {nt_xent(z, torch.randn(4, 8)):.3f} "
      f"| nếu mọi độ tương đồng bằng nhau: ln(2N − 1) = {math.log(7):.3f}")

## 2. Huấn luyện SimCLR trên 20 000 ảnh không nhãn

Encoder: 3 khối conv + BatchNorm + global average pooling → $h \in \mathbb R^{128}$; projection head: MLP 128 → 128 → 64. Batch 256 (mỗi ảnh có 510 "ứng viên âm"), 2 000 bước. Để so sánh, train thêm một bản **chỉ dùng augmentation độ sáng + nhiễu** (bỏ xoay/dịch/lật).

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        def block(i, o):
            return [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU()]

        self.net = nn.Sequential(*block(1, 32), nn.MaxPool2d(2), *block(32, 64), nn.MaxPool2d(2), *block(64, 128),
                                 nn.AdaptiveAvgPool2d(1), nn.Flatten())

    def forward(self, x):
        return self.net(x)


def train_simclr(geometric=True, steps=2000, tau=0.2, seed=0):
    torch.manual_seed(seed)
    encoder = Encoder().to(device)
    head = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 64)).to(device)
    opt = torch.optim.Adam(list(encoder.parameters()) + list(head.parameters()), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for step in range(steps):
        xb = X_unlabeled_t[torch.randint(0, len(X_unlabeled_t), (256,), generator=g)]
        v1, v2 = augment(xb, g, geometric).to(device), augment(xb, g, geometric).to(device)
        opt.zero_grad()
        loss = nt_xent(head(encoder(v1)), head(encoder(v2)), tau)
        loss.backward()
        opt.step()
    return encoder.eval(), loss.item()


encoders = {}
torch.manual_seed(0)
encoders["encoder ngẫu nhiên"] = Encoder().to(device).eval()
for name, geometric in [("SimCLR", True), ("SimCLR không aug hình học", False)]:
    t0 = time.perf_counter()
    encoders[name], final_loss = train_simclr(geometric)
    print(f"{name:26s}: loss cuối = {final_loss:.3f} (đoán mò: ln 511 = {math.log(511):.2f}) | {time.perf_counter() - t0:.0f}s")

Bản không có augmentation hình học có loss **thấp hơn** — bài toán dễ hơn (hai góc nhìn gần như giống hệt, chỉ khác độ sáng/nhiễu: chỉ cần nhớ "vị trí và kích thước" là ghép đúng). Loss thấp không có nghĩa là biểu diễn tốt — phải đánh giá trên tác vụ đích.

## 3. Đánh giá bằng linear probe khi chỉ có 50 hoặc 500 nhãn

Đóng băng encoder, lấy $h$ (bỏ projection head — $h$ thường tốt hơn $z$ cho tác vụ sau), train logistic regression với $n$ ảnh có nhãn; 3 lần chọn mẫu. So với: **pixel** thô, **encoder ngẫu nhiên**, **autoencoder** (bài 01, latent 32 chiều), và **train có giám sát từ đầu** cùng kiến trúc với đúng $n$ nhãn đó.

In [ ]:
@torch.no_grad()
def features(encoder, X):
    return torch.cat([encoder(X[i:i + 1000].to(device)).cpu() for i in range(0, len(X), 1000)]).numpy()


def linear_probe(F_pool, F_test, n, seeds=range(3)):
    scores = []
    for s in seeds:
        idx = np.random.default_rng(s).choice(len(F_pool), n, replace=False)
        clf = LogisticRegression(max_iter=5000).fit(F_pool[idx], y_hidden[idx])
        scores.append((clf.predict(F_test) == y_test).mean())
    return np.mean(scores)


class ConvAE(nn.Module):                                               # autoencoder của bài 01
    def __init__(self, k=32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Conv2d(1, 32, 3, 2, 1), nn.ReLU(), nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(), nn.Flatten(), nn.Linear(64 * 7 * 7, k))
        self.decoder = nn.Sequential(nn.Linear(k, 64 * 7 * 7), nn.ReLU(), nn.Unflatten(1, (64, 7, 7)),
                                     nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(), nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid())


torch.manual_seed(0)
ae = ConvAE().to(device)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(3000):
    xb = X_unlabeled_t[torch.randint(0, len(X_unlabeled_t), (128,), generator=g)].to(device)
    opt.zero_grad()
    F.mse_loss(ae.decoder(ae.encoder(xb)), xb).backward()
    opt.step()
encoders["autoencoder (bài 01)"] = ae.encoder.eval()

POOL = 5000
results = {"pixel thô": {n: linear_probe(X_unlabeled[:POOL].reshape(POOL, -1), X_test.reshape(len(X_test), -1), n) for n in [50, 500]}}
for name, enc in encoders.items():
    results[name] = {n: linear_probe(features(enc, X_unlabeled_t[:POOL]), features(enc, X_test_t), n) for n in [50, 500]}


def supervised_from_scratch(n, seeds=range(3)):
    scores = []
    for s in seeds:
        idx = torch.tensor(np.random.default_rng(s).choice(POOL, n, replace=False))
        torch.manual_seed(s)
        model = nn.Sequential(Encoder(), nn.Linear(128, 5)).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        g = torch.Generator().manual_seed(s)
        X, y = X_unlabeled_t[idx], torch.tensor(y_hidden)[idx]
        for _ in range(600):
            model.train()
            b = torch.randint(0, n, (64,), generator=g)
            opt.zero_grad()
            F.cross_entropy(model(X[b].to(device)), y[b].to(device)).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            scores.append((model(X_test_t.to(device)).argmax(1).cpu() == torch.tensor(y_test)).float().mean().item())
    return np.mean(scores)


results["có giám sát từ đầu (n nhãn)"] = {n: supervised_from_scratch(n) for n in [50, 500]}
print(f"{'biểu diễn':30s} {'50 nhãn':>9s} {'500 nhãn':>9s}")
for name, r in results.items():
    print(f"{name:30s} {r[50]:9.3f} {r[500]:9.3f}")

- **SimCLR + linear probe** với 50 nhãn (10 ảnh/lớp) vượt xa mọi biểu diễn khác — và vượt cả mạng **có giám sát** train từ đầu trên chính 50 nhãn đó; với 500 nhãn gần như hoàn hảo. Toàn bộ "hiểu biết" về hình dạng đến từ 20 000 ảnh **không nhãn**.
- **Autoencoder** chỉ hơn pixel thô một chút — đúng như bài 01: latent của nó mã hoá vị trí/kích thước.
- **Bỏ augmentation hình học**, SimCLR kém hẳn: biểu diễn chỉ bất biến với những gì augmentation thay đổi. Augmentation chính là cách ta **mô tả** "thế nào là cùng một thứ" — chọn sai, biểu diễn học sai thứ.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, name in zip(axes, ["autoencoder (bài 01)", "SimCLR"]):
    Z = PCA(2).fit_transform(features(encoders[name], X_test_t))
    for c in range(5):
        ax.scatter(Z[y_test == c, 0], Z[y_test == c, 1], s=5, alpha=0.6, label=SHAPE_CLASSES[c])
    ax.set_title(f"{name} (PCA 2D) — tô màu theo nhãn ẩn")
    ax.legend(fontsize=8, markerscale=2)
plt.tight_layout()
plt.show()

## 4. Bức tranh rộng hơn

| Phương pháp | Ý tưởng | Ghi chú |
|---|---|---|
| SimCLR, MoCo | tương phản: kéo cặp dương lại gần, đẩy cặp âm ra xa | cần nhiều mẫu âm (batch lớn hoặc hàng đợi) |
| BYOL, SimSiam | chỉ dùng cặp dương + mạng dự đoán + dừng gradient / mạng giáo viên trung bình động | không cần mẫu âm; tránh **sụp đổ** (mọi ảnh cùng một vector) nhờ bất đối xứng |
| DINO, DINOv2 | tự chưng cất (self-distillation) học sinh–giáo viên trên ViT | đặc trưng ảnh tốt nhất hiện nay cho linear probe |
| MAE (Masked Autoencoder) | che 75% patch của ViT, tái tạo phần bị che | "BERT cho ảnh" — tái tạo nhưng phải **suy luận** phần thiếu, không chỉ sao chép |
| CLIP | tương phản **ảnh ↔ văn bản** trên 400 triệu cặp | zero-shot: phân loại bằng mô tả văn bản; nền tảng của nhiều mô hình đa phương thức |
| Ngôn ngữ (chương 31–32) | dự đoán từ bị che / từ tiếp theo | word2vec, BERT, GPT đều là tự giám sát |

**Sụp đổ biểu diễn** (collapse): nếu chỉ kéo cặp dương lại gần, nghiệm tầm thường là ánh xạ mọi ảnh vào cùng một điểm (loss = 0). Mẫu âm (SimCLR), bất đối xứng + dừng gradient (BYOL/SimSiam), hay chuẩn hoá phương sai (VICReg) là các cách ngăn điều đó.

Trong thực tế hiếm khi tự train self-supervised từ đầu: dùng **embedding từ mô hình nền tảng** (CLIP, DINOv2, sentence-transformers — chương 32) rồi linear probe / kNN / tìm kiếm tương tự. Tự train có ý nghĩa khi có rất nhiều dữ liệu **miền riêng** không nhãn (ảnh y tế, vệ tinh, công nghiệp) mà mô hình nền tảng chưa thấy.

## 🧠 Kiểm tra nhanh

1. Học tự giám sát khác học không giám sát "cổ điển" (phân cụm, PCA) thế nào?
2. Mô tả SimCLR và loss NT-Xent. Nhiệt độ τ có vai trò gì?
3. Vì sao loss contrastive thấp hơn không đồng nghĩa với biểu diễn tốt hơn?
4. Vì sao biểu diễn SimCLR phân loại hình tốt hơn autoencoder?
5. "Sụp đổ biểu diễn" là gì? Các phương pháp tránh nó thế nào?
6. Khi nào nên tự train self-supervised thay vì dùng mô hình nền tảng có sẵn?

## 📌 Tóm tắt

- Self-supervised tạo nhãn từ chính dữ liệu; biểu diễn học được phản ánh **mục tiêu** và **augmentation**.
- SimCLR: hai góc nhìn tăng cường là cặp dương, InfoNCE phân biệt chúng giữa các ảnh khác; linear probe với vài chục nhãn đã đạt độ chính xác cao.
- Augmentation định nghĩa bất biến; mục tiêu tái tạo pixel ưu tiên thông tin không cần thiết.
- Họ phương pháp: tương phản, không mẫu âm (BYOL/SimSiam/DINO), che-và-tái-tạo (MAE), đa phương thức (CLIP).

## 📚 Tài liệu tham khảo

- Chen et al. (2020) — *A Simple Framework for Contrastive Learning of Visual Representations* (SimCLR)
- van den Oord, Li & Vinyals (2018) — *Representation Learning with Contrastive Predictive Coding* (InfoNCE)
- He et al. (2020) — *Momentum Contrast* (MoCo); Grill et al. (2020) — *BYOL*; Chen & He (2021) — *SimSiam*
- Caron et al. (2021) — *DINO*; He et al. (2022) — *Masked Autoencoders Are Scalable Vision Learners*
- Radford et al. (2021) — *Learning Transferable Visual Models From Natural Language Supervision* (CLIP)
- Balestriero et al. (2023) — *A Cookbook of Self-Supervised Learning*